# Notebook for the house prices advanced regression kaggle competition

## Loading the data and the modules

In [1]:
# Models
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
# Model and feature selection tools
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.feature_selection import SelectFromModel
# Pipeline and data transformation tools
from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, TargetEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import root_mean_squared_error

# Usual packages
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

valid = pd.read_csv('test.csv')
init_train = pd.read_csv('train.csv')

## Defining the columns that will be treated

In [2]:
# Replacing most of the NaN with the actual value it should have according to the data description
replaceNA = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageQual', 'GarageFinish', 'GarageType', 'GarageCond', 'BsmtFinType2', 'BsmtExposure', 'BsmtCond', 'BsmtQual', 'BsmtFinType1']
replaceNone = ['MasVnrType']

# Columns to remove : Found after exploring the dataset
removed_cols = ['Id', 'LotFrontage', 'GarageYrBlt', 'MasVnrArea']

# Encoding categorical features that have ordered ordinal values (such as quality) or that could be boolean
cols_mapped = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "HeatingQC", "CentralAir", "KitchenQual", "Functional", "FireplaceQu", "GarageFinish", "GarageQual", "GarageCond", "PoolQC"]
mappings = {
    "ExterQual": {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex":4},
    "ExterCond": {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex":4},
    "BsmtQual": {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "BsmtCond": {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "BsmtExposure": {"NA": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2": {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "HeatingQC": {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex":4},
    "CentralAir": {"N": 0, "Y": 1},
    "KitchenQual": {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex":4},
    "Functional": {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
    "FireplaceQu": {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageFinish": {"NA": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "GarageQual": {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageCond": {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "PoolQC": {"NA": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4}
}

categories = [[k for k, v in sorted(mappings[col].items(), key=lambda x: x[1])] for col in cols_mapped]

## First cleaning of the data
Removing 2 outliers and a row with missing values

In [3]:
filtred_train = init_train.drop(init_train.loc[init_train["Electrical"].isnull()].index, axis=0)
print(f"Number of remaining missing values : {filtred_train.isnull().sum().sum()}")
filtred_train = filtred_train.drop([523, 1298], axis=0)

Number of remaining missing values : 7822


## Pipeline for applying data cleaning and training

In [ ]:
# Splitting data
filtred_X, filtred_y_log = filtred_train.drop(["SalePrice"], axis=1), np.log1p(filtred_train.SalePrice)
X_train, X_test, y_train, y_test = train_test_split(filtred_X, filtred_y_log, test_size=0.2, random_state=1)

# Getting categorical features' cardinality
num_cols = X_train.select_dtypes(include='number').columns
obj_cols = X_train.columns.difference(num_cols)
obj_cardinality = {}
for objcol in obj_cols:
    obj_cardinality[objcol] = len(X_train[objcol].unique())

# dividing our categorical features into low and high cardinality ones using a cardinality threshold
high_cardinality_threshold = 5
obj_low_card = [objcol for objcol in obj_cols if obj_cardinality[objcol] < high_cardinality_threshold]
obj_high_card = [objcol for objcol in obj_cols if obj_cardinality[objcol] >= high_cardinality_threshold]

def fill_categorical_na(df, cols_na, cols_none):
    df = df.copy()
    df[cols_na] = df[cols_na].fillna("NA")
    df[cols_none] = df[cols_none].fillna("None")
    return df
fill_na_step = FunctionTransformer(fill_categorical_na, kw_args={"cols_na": replaceNA, "cols_none": replaceNone})

# Creating our preprocessing step
preprocessor = ColumnTransformer([
    ("to_drop", "drop", removed_cols),
    ("num", "passthrough", num_cols),
    ("nom_as_described", OrdinalEncoder(categories=categories, handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.int64), cols_mapped),
    ("nom_low", OneHotEncoder(handle_unknown="ignore", sparse_output=False), obj_low_card),
    ("nom_high", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), obj_high_card),
])
# ("nom_high", TargetEncoder(target_type="continuous", cv=5), obj_high_card)

# Creating our pipeline for preprocessing, feature selection and training
pipes = [
    Pipeline([
    ("fill_na", fill_na_step),
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(estimator=XGBRegressor(n_estimators=200, max_depth=6))),
    ("model", XGBRegressor(n_estimators=100))
]),
    Pipeline([
    ("fill_na", fill_na_step),
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(estimator=RandomForestRegressor(n_estimators=200, max_depth=6))),
    ("model", RandomForestRegressor())
])
]

# Definition of the hyperparameters to tune
param_grid = [{
    "feature_selection__threshold": ["median", "mean", 0.1, 0.05, 0.01],
    "model__max_depth": [3, 5, 7, 10, 20],
    "model__learning_rate": [0.01, 0.05, 0.1]
},
{
    "feature_selection__threshold": ["median", "mean", 0.05, 0.01],
    "model__n_estimators": [50, 100, 200, 300],
    "model__max_depth": [3, 5, 7, 10, 20],
    "model__min_samples_leaf": [1, 2, 3]
}
]

inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
grids = []
nested_scores = []
for model_ite in range(len(pipes)):
    # Inner loop : Used to define our best hyperparameters
    grids.append(GridSearchCV(estimator=pipes[model_ite], param_grid=param_grid[model_ite], cv=inner_cv, scoring="neg_root_mean_squared_error", n_jobs=-2, return_train_score=True))
    # Outerloop : Used to loop our hyperparameter search to reduce the uncertainty
    nested_scores.append(cross_val_score(grids[model_ite], X_train, y_train, cv=outer_cv, scoring="neg_root_mean_squared_error"))

print("Scores : ")
print(f"XGB : {-nested_scores[0].mean()} +- {nested_scores[0].std()}")
print(f"RF : {-nested_scores[1].mean()} +- {nested_scores[1].std()}")


Scores :

XGB : 0.1237243194814969 +- 0.010729953942546812

RF : 0.13666735372139388 +- 0.01019826749097294

In [26]:
# Selecting the best model
best_model_idx = 0 if nested_scores[0].mean() > nested_scores[1].mean() else 1

# Looking for the best hyperparameters and training the model
final_grid = GridSearchCV(pipes[best_model_idx], param_grid[best_model_idx],
                           cv=KFold(n_splits=5, shuffle=True, random_state=42),
                           scoring="neg_root_mean_squared_error", n_jobs=-2)
final_grid.fit(X_train, y_train)

# Testing the model with our test sample
y_pred = final_grid.predict(X_test)
test_rmse = root_mean_squared_error(y_test, y_pred)
print(f"RMSE final sur test set : {test_rmse}")

RMSE final sur test set : 0.12309631764622213


## Engineering features

In [46]:
print(f"XGB : {-nested_scores[0].mean()} +- {nested_scores[0].std()}")
print(f"RF : {-nested_scores[1].mean()} +- {nested_scores[1].std()}")


XGB : 29731.705078125 +- 2593.5575949606346
RF : 32032.6735667538 +- 2159.576037949638


## Competition prediction

In [ ]:
# Training the model with the best hyperparameters with the full training set
best_params = final_grid.best_params_
final_pipeline = pipes[best_model_idx].set_params(**best_params)

final_pipeline.fit(filtred_X, filtred_y_log)

In [ ]:
# Making the predictions with the test data
valid_ids = valid["Id"]

y_pred_log = final_pipeline.predict(valid)
y_pred = np.expm1(y_pred_log)

submission = pd.DataFrame({
    "Id": valid_ids,
    "SalePrice": y_pred
})
submission.to_csv("submission.csv", index=False)